# 2026/8/15

# 模型微调

- 目标：

  在预训练阶段，模型在大规模标注文本上学习对下一个token的分布表示；在下游应用中，通过微调（fine-tuning）可使模型适应特定任务领域或风格（如问答、代码生成、对话系统等）。最核心的是学会指令遵循，把知识按照要求表达出来

- 应用场景
  
  - 回答风格调整：模拟特定语气回答，针对用户体验感
  - 问答对记忆：指定知识进行回答
  - 领域知识灌注：同上
  - 代码/数学能力增强：模型刷榜前进行
  - function calling 能力增强
  - agent 能力增强：RAG 系统中使用一个微调过小模型增强路由能力

<div align="center">

<img src="note_pics/post-train.png">
<br>
大模型开发流程

</div>

> 这里值得一提的是，RL 有时用在 **给模型赋予“拒绝回答”某类问题的能力**，比如说一些可能侵犯别人安全的行为。相比于 SFT ，这个步骤使用强化学习更能与人类偏好对齐**

- 与预训练的区别

  - 数据格式不同
    
    预训练就是把文本一条一条直接输入给模型；而微调是赋予模型与人对话的能力，因此需要构造问答对数据集，针对不同的模型有不同的 chat template

  - 损失计算方式不同

    - 预训练中，输入文本的每个内容都需要进行损失计算；对于微调这部分叫 completion only ，也就是 **只对于模型回答部分** 进行损失计算，不考虑用户输入部分。

    - 实现方式为使用 loss mask，将特定 token 标志（如 ```<end_header_id>```）位置开始的 mask 设置为1，其余部分设置为 0。如下图所示

      <div align="center">

      <img src="note_pics/loss-mask.png">

      </div>

    - 下面实现 loss mask，原文件位于```llm_study\week4\课程内容\手撕微调（原理篇）\train_sft\sft_completions_only.py```
      
      > 这里我一开始想问：```max_length``` 值与 ```input_len``` 是否一样？因为它设置了 ```padding=True```
      
      > 答：```max_length``` 是每条文本截断的上限；而 ```padding=True``` 默认会补齐到“当前 batch 里最长那条”的长度，因此代码中 ```input_len=min(max_length, 当前 batch 截断后最长样本的长度)```

In [ ]:
def sft_collate(batch, tokenizer, end_str, max_length):
    end_str = "<|start_header_id|>assistant<|end_header_id""|>\n\n"

    # 获取该批文本的 token_id 以及统一长度
    inputs = tokenizer(batch, max_length=max_length, padding=True, truncation=True)
    input_ids = inputs["input_ids"]
    input_len = len(input_ids[0])

    end_ids = tokenizer(end_str)["input_ids"]
    end_id_len = len(end_ids)

    # 单个 loss_mask 的总长度应该与 input_len 长度相同
    # loss_mask 的总数量与 batch 大小相同
    loss_mask = []
    for input_id in input_ids:
        
        # 针对每一个问答对的 token_id 倒序查找，滑动窗口匹配，是否窗口中的内容是模型开始回答的标志
        for i in range(len(input_id) - end_id_len, -1, -1):

            if input_id[i:i + end_id_len] == end_ids:
                mask = [1] * (input_len - 1)
                mask[:i + end_id_len - 1] = [0] * (i + end_id_len - 1)
                loss_mask.append(mask)
                break

            if i == 0:  # 所有回答部分都被截断
                loss_mask.append([0] * (input_len - 1))

    inputs = {k: torch.tensor(v) for k, v in inputs.items()}
    
    """
    上面操作将 inputs 中每个键对应的值都转化为 torch 张量，方便后续直接使用 inputs["input_ids"] 进行调用
    等价于：

    new_inputs = {}

    for k, v in inputs.items():
        new_inputs[k] = torch.tensor(v)

    inputs = new_inputs
    """

    loss_mask = torch.tensor(loss_mask)
    return inputs, loss_mask

  -  NEFTune (Noisy Embedding Instruction Fine Tuning)

      <div align="center">

      <img src="note_pics/NEFTune.png">

      </div>

      - 类似于 CV 领域中给训练图片进行不同种类的旋转、增强，从而提升模型泛化能力。这个方法在模型微调过程中，通过给 Embeddeding 向量增加一个均匀噪声，提升了模型的表达能力。
        
        > AlpacaEval 可以理解成一个专门用来评估 大语言模型“回答问题好不好” 的评测基准。评测结果是一个百分数，代表回答被判断为优质回答的比例

      - 同一个 token 的向量经过扰动后表达的意思可以说是几乎不变，类比成给图片切割、旋转后进行训练

        <div align="center">

        <img src="note_pics/word-embedding.png">

        </div>

      - 下面是手撕 NEFTune（这个功能在 Huggingface 包中已经内置好了，只是一个超参数）




In [ ]:
neftune_noise_alpha = 10    # 一般取值 0-10 之间

for i in range(epoch):
    for inputs, loss_mask in data_loader:

        input_ids = inputs.pop("input_ids") # [bs, len]

        # 获取 token embedding 
        input_embeddings = model.base_model.model.embed_tokens(input_ids)   # [bs, len, dim]

        # 计算 embedding 总维度，即多少个位置需要加噪声
        dims = torch.tensor(
            input_embeddings.size(1) * input_embeddings.size(2)
        )

        # 计算噪声幅度
        mag_norm = neftune_noise_alpha / torch.sqrt(dims)

        # 生成均匀分布噪声并添加到 embedding
        input_embeddings = (
            input_embeddings
            + torch.zeros_like(input_embeddings).uniform_(
                -mag_norm,
                mag_norm
            )
        )

        inputs["inputs_embeds"] = input_embeddings

<div style="page-break-after: always;"></div>

# 2026/8/17

## 微调方式

### 0. 全量微调/全参数微调 (Full Fine-tuning) 

- 顾名思义，使用构造好的问答对对原本模型所有参数进行更新

- 由于全量微调计算和存储成本高昂，高效微调算法 (Parameter-Efficient Fine-tuning, PEFT) 应运而生，旨在以极低的参数调整代价，实现媲美全参数微调的性能。
  
  下面是一些 PEFT 方法

### 1. prompt-tuning

- Prompt-Tuning 的思想

  流结主模型全部参数，在训练数据前加入一个小段Prompt，只训练Prompt的表示层，即一个Embedding模块。

<div align="center">

<img src="note_pics/prompt-tunning.png">

</div>

- 方式

  - hard prompt

    在微调过程中给每个种任务之前加入对应内容的提示词，这部分提示词内容与任务本身有关，是可以被单独设计、可读的。此时模型更新的参数为模型本身。

  - soft prompt

    微调时候给每个输入的 Embedding 张量前面再加上一个**可以训练的 prompt embedding**，相当于为所有输入都设计了一个共有的提示词方便模型理解问题。这里模型更新的参数只是拼接上的提示词的嵌入向量。

### 2. prefix-tuning

- past_key_values
  
  Transformer 计算过程中，会存在大量的重复计算，因此可以将 key 和 value 的计算结果缓存，作为 past_key_values 输入到下一次的计算中，这一技术又被称之为 kv_cache

- Prefix-Tuning 思想

  通过 past_key_values 的形式将学习的部分放到了模型中的每一层，这部分内容又被称为前缀。与 soft prompt 这样**针对输入侧**的算法不同的是，这个方法**在每一层都有一个相当于额外输入的 KV 前缀**用于让模型更新参数，从而理解问答关系。

<div align="center">

<img src="note_pics/prefix-tuning.png">

</div>

- 实现方式

  通常先学习一个较小的前缀 $P$ ，然后通过一个 Prefix Encoder 映射（原文使用MLP）：

  $$
  P \rightarrow K_{prefix},V_{prefix}
  $$

  因此微调过程中大模型完全冻结，只训练 Prefix

### 3. adapter

- 核心思想
  
  在Transformer的注意力层和前馈神经网络（FFN）层之后添加全连接网络。微调时，只对新增的Adapter结构和Layer Norm层进行微调，从而保证了训练的高效性。每当出现新的下游任务，通过添加Adapter模块产生一个易于扩展的下游模型，从而避免全量微调导致的灾难性遗忘的问题

  <div align="center">
  
  <img src="note_pics/adapter.png">
  
  </div>

- 实现方式
  
  Adapter 的内部实现正好与 FFN 相反，它是将向量先降维再升维，从而让模型理解特定的任务。而其内部又有残差链接，所以模型本身通用能力能够得到保留。

  $$
  h'=h+\underbrace{W_{up}\sigma(W_{down}h)}_{\text{任务特定的修正}}
  $$
  
  训练时仅更新每层 adapter 的参数

### 4. LoRA 微调

  - 核心思想
    
    LoRA (Low-Rank Adaptation) 是一种基于低秩分解 (Low-Rank Decomposition) 的高效微调算法。其核心思想是冻结预训练模型权重，只训练少量低秩适配器 (Low-Rank Adapters)，通过模拟权重更新的方式，实现模型微调。

    <div align="center">
    
    <img src="note_pics/lora.png">
    
    </div>

  - 实现方式

    LoRA 矩阵一般加在**模型的 Linear 层**，如 Transformer中的 $W_q、W_k、W_v、W_o$，FFN 中的 $W_{gate}、W_{up}、W_{down}$

  - 数学原理

    模型在微调后需要改变的参数为

    $$
    \Delta W=BA
    $$

    其中，

    $$
    A\in R^{r\times d_{in}}
    $$
    $$
    B\in R^{d_{out}\times r}
    $$

    > 值得一提的是，$A$ 矩阵随机初始化，$B$ 矩阵初始化为 0 矩阵，所以一开始微调的时候模型总的输出和原本一样，之后才进行增量更新。

  - 缩放尺度

    $$
    W'=W+\frac{\alpha}{r}BA
    $$

    这个 $\alpha/r$ 本质上就是控制 LoRA 更新量的强弱，让 rank 改变时，LoRA 对原模型的影响保持在一个比较合理、稳定的范围。

## 微调质量

### 业界共识

1. prompt的**质量和多样性**远重要于数据量级，微调一个30b量级的base model只需要 10w 量级的数据即可。对于一个领域，尽量使数据集的 prompt 覆盖到更多的使用情况，一种情况一条就够了。

  这一点同样适用于数据构建上，也是在简历中的亮点。参考：《LIMA: Less Is More for Alignment》

2. 合成数据很重要！——一般需要通过不同方式进行多路合成，减少合成数据的bias。

  参考：《Phi-3 Technical Report: A Highly Capable Language Model Locally on Your Phone》

3. 可以加点**预训练的数据**进去，减少灾难性遗忘现象。

  参考：《The Llama 3 Herd of Models》

4. 一般训练一个epoch，垂直领域模型数据少的话训练3epoch去过拟合。

5. 可以全量微调，就不要做PEFT。

6. SFT阶段不能太多知识注入，过多的知识注入，或者超出模型能力本身的回答过多会导致对齐税

### 数据合成

- 合成Prompt

1. 划分技能库，给每一个数据打上 tasktype，越细越好

2. 随机采样seed，结合第一点，从不同数据库中抽取seed（可以分层抽样），造新prompt合成数据

3. 纯指令生成prompt，用启发式的规则过滤

4. 合成不同格式的（如 markdown，json），行文风格的 prompt 合成 Answer

- 指令参考 

1. 英文数据，GPT4 is all you need

2. 中文模型，Qwen_72B / deepseek_MOE

3. 给定约束条件

4. 尽量给CoT，如果需要模型具有特定的推理能力的话

- 工业界做法

Train 一个 reward model 或者用规则拒绝采样（打分），选出合适的 SFT/DPO 训练数据

#### Adam 优化器以及其参数

- Adam 更新参数方式：**$m_t$ 决定更新方向， $v_t$ 决定更新步长， 把当前的梯度趋势 $m_t$ 按照历史梯度大小 $v_t$ 做归一化**，从而更新网络参数

  表达式如下：
  
  $$
  \theta_{t+1}=\theta_t-\eta\frac{m_t}{\sqrt{v_t}+\epsilon}
  $$
  
  $$
  m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
  $$
  
  $$
  v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2
  $$

  - $m_t$
  
    - 从数学角度看，每次网络参数更新的方向为 **上一次梯度的方向** 与 **本次梯度方向** 加权求和的结果
  
    - $m_t$ 每次都把 **过去梯度的综合结果和当前梯度** 按照权重重新平均，而且越久远的梯度权重按照 $β_1^n$ 指数衰减。
    
    - 根据表达式内容，如果过去的梯度有一个统一的趋势，比如都是正的，则接下来梯度的更新也应该是这个趋势
  
  - $v_t$
    - 从数学角度看，每次网络参数更新的大小为 **上一次梯度的大小** 与 **本次梯度大小** 加权求和的结果
  
    - $v_t$ 表示 **最近一段时间这个参数的梯度强度**，注意表达式中是平方项，相当于模长平方

- 参数设置一般如下：

  ```
  Adam(
      lr=1e-3,
      betas=(0.9, 0.999),
      eps=1e-8
  )
  ```
  betas 中分别代表一阶、二阶动量的衰减系数

- **注意**：

  即使模型在训练时候用的精度是 BF16 ，其所用 **Adam 优化器的精度应该保持 FP32**

  这是因为：参数更新要求数值稳定，而 Adam 的 $m, v$ 是长期累积的统计量，对精度非常敏感。

  模型参数 $W$ 虽然也需要更新，但训练过程中通常会使用较高精度进行计算，并且 BF16 的动态范围非常大（约等于 FP32），适合模型参数记录。

### 计算机的浮点数表示法

  在 IEEE 754 浮点数里，正常数可以写成：

  $$
  1.M\times 2^E
  $$

  这里的 $M$ 就是尾数部分，如

  $$
  1.1011_2\times2^3
  $$

  其中的二进制表示

  $$
  1.1011_2=1+\frac12+\frac18+\frac1{16}=1+0.5+0.125+0.0625=1.6875
  $$

  所以原式

  $$
  1.6875\times2^3=13.5
  $$
  
  因此一个 BF16 的数值
  
  $$
  1.1010101_2\times2^E
  $$

  > BF16 精度比 FP16 低的根本原因：BF16 只有 7 个显式尾数位，FP16 有 10 个显式尾数位。

  对于指数项，BF16 指数位为 8 ，则 $2^8-1=255$；为了让指数可以表示正、负，则将 256 均分

  根据浮点数表达式可以推导，对于一个 BF16 数值
  
  - 正常数最大是 $(2-2^{-7})\times2^{127}$ 约等于 ${3.39\times10^{38}}$ （括号中的是等比数列求和结果）

  - 正常数最小是 $1.0\times2^{-126}$ 约等于 $1.17\times10^{-38}$

  <div align="center">

  | 类型       | 符号位 |   指数位 |    尾数位 | 总位数 |
  | -------- | --: | ----: | -----: | --: |
  | FP32     |   1 |     8 |     23 |  32 |
  | **BF16** |   1 | **8** |      7 |  16 |
  | **FP16** |   1 | **5** | **10** |  16 |
  
  <br>
  
  |       |                 BF16 |                FP16 |
  | ----- | -------------------: | ------------------: |
  | 符号位   |                    1 |                   1 |
  | 指数位   |                **8** |               **5** |
  | 尾数位   |                    7 |              **10** |
  | 最大值   |  $(3.39\times10^{38})$ |    $(6.55\times10^4)$ |
  | 最小正常值 | $(1.17\times10^{-38})$ | $(6.10\times10^{-5})$ |

  </div>


<div style="page-break-after: always;"></div>

# 2026/8/18

## 显存计算和量化

### 显存计算

> **模型参数与占用显存估计** 内容详见 ```llm_study\week3\notes\Llama笔记.ipynb```

模型训练并不只是把模型加载到显卡上就够了，每个参数除了自身数值以外，都有一个自身的梯度数值和优化器结果（$m, v$ 两个数值）

所以，一个 Llama-13B 模型，假设使用 BF16 保存模型参数，则估算它在训练过程中占用显存大小（除去激活值）：

$$
模型自身参数：13 \times 2 = 26 GB
$$
$$
梯度参数：13 \times 2 = 26 GB
$$
$$
Adam 优化器数值 (FP32)：13 \times 4 \times 2 = 104 GB
$$

**总共约 156GB**

<div align="center">

<img src="note_pics/显存占用.png">

</div>

- 激活值 (Activation) 占用显存
  
  - 为什么需要激活值

    $x$ 输入；$w_1,w_2$ 模型参数；$z_1,a_1,z_2$ 前向传播产生的中间结果；$y$ 真实标签
  
    $$
    z_1=xw_1
    $$
    $$
    a_1=\sigma(z_1)
    $$
    $$
    z_2=a_1w_2
    $$
    $$
    Loss=(y-z_2)^2
    $$
  
    反向传播需要更新 $w_1,w_2$，也就是需要计算 $\frac{\partial Loss}{\partial w_1}$ 和 $\frac{\partial Loss}  {\partial w_2}$
    其中，
  
    $$
    \frac{\partial Loss}{\partial w_2}=\frac{\partial Loss}{\partial z_2}\frac{\partial z_2}{\partial   w_2}=2(z_2-y)a_1
    $$
    
    $a_1$ 是前向传播产生的结果
    
    $$
    \frac{\partial Loss}{\partial w_1}=2(z_2-y) \times w_2 \times \sigma(z_1)(1-\sigma(z_1)) \times x
    $$
  
    这里需要 $z_1$ 和 $x$
  
    由此可见，**反向传播的链式法则需要前向传播过程中产生的中间结果**，因此这些中间结果需要保存下来，这些就是   Activation 激活值，如例子中的 $x,z_1,a_1,z_2$
  
  - 激活值占用估计（默认使用 BF16/FP16）
    
    公式重在理解比例关系即可：

    $$
    M_{\text{activation}}=\frac{sbh\left(34+5a\frac{s}{h}\right)L}{1024^3}
    $$

    <div align="center">
    
    | 符号  | 含义                 |
    | --- | ------------------ |
    | $s$ | sequence length    |
    | $b$ | batch size         |
    | $h$ | hidden size        |
    | $a$ | attention heads 数量 |
    | $L$ | Transformer 层数     |
    
    <br>
    </div>

    - 34 是一个经验系数，每一层 Transformer 中，大约有 34 个 b×s×h 规模的 Activation 需要保存（$X, Q, K, V$ 以及各种线性层、残差、Norm 等中间结果 还有 FFN 部分）

    - 5 是一个关于 Attention 计算的经验系数，表示 Attention 中与 $S^2$ 相关的 Activation 显存

    - 一个 Llama-13B 模型，$s=1024, b=1, h=5120, a=40, L=40$ 激活值占用显存为 14.5GB，所以说如果想要实现现在大模型那种 1M 上下文，训练时需要的显存占用非常大
      > 这里需要注意的是，序列长度是 **一次模型输入和输出的长度** ，不是输入给模型的序列长度

- 综上所述，一个 Llama-13B 模型，设置 Batchsize=1, Seq_len=1024, FP16 进行全量微调，占用总显存
  
  $$
  模型自身参数：13 \times 2 = 26 GB
  $$
  $$
  梯度参数：13 \times 2 = 26 GB
  $$
  $$
  Adam 优化器数值 (FP32)：13 \times 4 \times 2 = 104 GB
  $$
  $$
  激活值：14.5 GB
  $$

  合计：222.5 GB。一张 H100 80GB，所以需要 4 张 H100

- LoRA 微调 **节省模型参数部分显存，原模型激活显存不变**
  
  > LoRA 微调相当于给每一层的 Liner 又加一个由两个小矩阵组成的低秩分支，输入同时经过这两层，但是原本的那部分参数不更新，只更新后来加上去的那两个小矩阵

  前向传播过程中，表达式后一项就是 LoRA 对原模型输出的修正

  $$
  Y=XW+XAB
  $$

  现在 Loss 产生一个梯度 $\frac{\partial L}{\partial Y}$，为了更新 $A, B$，反向传播必须知道前向过程中的每一层的 $X$，因为 $\frac{\partial L}{\partial B}$ 里面就包含 $X$

  > LoRA 大幅减少的是参数、梯度和优化器状态显存，而不是激活显存。LoRA 微调显存占用量大约是模型参数量乘2（如 13B 模型做LoRA微调大约需要 26GB 显存）；全量微调显存占用大约是模型参数量乘 20